# 2.3 · 大数定律 & 中心极限定理（深挖）/ LLN & CLT, Deeper

> **课程定位**
> 0.9 节看过 CLT 的四面板动画。本课往深处走三层：**收敛多快（标准误）、什么统计量服从 CLT（中位数也行！最大值不行）、CLT 什么时候彻底失效（Cauchy 惊悚案例）**。这些直接决定 2.5 置信区间和 2.6 检验的适用边界。
> Three layers deeper than 0.9: how fast (standard error), which statistics obey CLT (medians yes, maxima no), and when CLT fails outright (the Cauchy horror story).

> 💡 **面试相关**
> - "标准差 vs 标准误的区别" ★★★★★（必考，答错直接挂）
> - "样本量翻倍，精度提高多少" ★★★★（$\sqrt{2}$，不是 2）
> - "CLT 对偏态数据要多大 n 才够" ★★★
> - "什么情况 CLT 失效" ★★★

---

## 目录
1. [LLN：两个版本与收敛速度](#1)
2. [⭐ 标准误：统计学最重要的公式](#2)
3. [√n 法则：精度的代价](#3)
4. [CLT 对偏态数据：n 要多大](#4)
5. [不止均值：中位数 / 分位数的 CLT](#5)
6. [⚠ CLT 失效案例 1：最大值（极值理论）](#6)
7. [⚠ CLT 失效案例 2：Cauchy（无均值）](#7)
8. [实战：模拟器验证一切](#8)
9. [小结](#9)


<a id="1"></a>
## 1. LLN：两个版本与收敛速度 / Two LLNs & Convergence Rate

| | 弱大数定律 (WLLN) | 强大数定律 (SLLN) |
|---|---|---|
| 表述 | $\bar{X}_n \xrightarrow{p} \mu$（依概率）| $\bar{X}_n \xrightarrow{a.s.} \mu$（几乎必然）|
| 人话 | 任取精度 ε，$n$ 大了之后"偏离超过 ε"的**概率**趋 0 | 单条样本路径**本身**终将贴住 μ 不再离开 |
| 要求 | $\mathbb{E}\lvert X \rvert < \infty$ | 同左 |

**收敛速度**由 Chebyshev 给出界：
$$\Pr(\lvert\bar{X}_n - \mu\rvert > \epsilon) \le \frac{\sigma^2}{n\,\epsilon^2}$$

误差以 $O(1/\sqrt{n})$ 速度收缩——这个速度就是下一节的标准误。


In [ ]:
import numpy as np
import scipy.stats as st
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)

# 单条样本路径的 running mean —— SLLN 的"几乎必然"长这样
# One sample path's running mean — what "almost surely" looks like
n = 100_000
x = rng.exponential(scale=2.0, size=n)          # μ = 2
running_mean = np.cumsum(x) / np.arange(1, n+1)

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(running_mean, lw=0.8)
ax.axhline(2.0, color="red", ls="--", label="μ = 2")
# 叠加 ±2 SE 包络 / overlay the ±2·SE envelope
ns = np.arange(1, n+1)
ax.fill_between(ns, 2 - 2*2/np.sqrt(ns), 2 + 2*2/np.sqrt(ns), alpha=0.15, color="red",
                label="μ ± 2σ/√n envelope")
ax.set_xscale("log"); ax.set_xlabel("n (log scale)"); ax.legend()
ax.set_title("LLN: running mean settles onto μ at rate 1/√n")
plt.tight_layout(); plt.show()


**红色包络（$\mu \pm 2\sigma/\sqrt{n}$）几乎一直罩住蓝线**——LLN 给方向，$1/\sqrt{n}$ 给速度。这条包络就是即将登场的主角。
The envelope hugs the path — LLN gives the destination, $1/\sqrt n$ gives the speed.


<a id="2"></a>
## 2. ⭐ 标准误：统计学最重要的公式 / The Standard Error

$$\boxed{\;\mathrm{SE}(\bar{X}) = \frac{\sigma}{\sqrt{n}} \;\approx\; \frac{s}{\sqrt{n}}\;}$$

### 标准差 vs 标准误 —— 必考题，一张表终结

| | 标准差 $\sigma$ (SD) | 标准误 $\mathrm{SE}$ |
|---|---|---|
| 描述谁的散布 | **单个观测**的散布 | **样本均值（估计量）**的散布 |
| 随 $n$ 变吗 | ❌ 不变（数据本身的性质）| ✅ 以 $1/\sqrt{n}$ 缩小 |
| 用途 | 描述数据、异常检测 | **置信区间、假设检验** |
| 例子 | "用户消费 std = \$50" | "均值估计的 SE = \$0.5（n=10000）" |

**一句话**：SD 描述数据，SE 描述"你对均值的不确定性"。报告均值时给 SE，描述人群时给 SD——**用反了是论文被拒的经典原因**。
SD describes the data; SE describes your uncertainty about the mean. Swapping them is a classic paper-rejection reason.


In [ ]:
# 直观验证: 把"样本均值"本身当随机变量看它的分布
# The sampling distribution of the mean, empirically
sigma, mu = 50.0, 100.0
for n in [25, 100, 400]:
    means = rng.normal(mu, sigma, size=(20_000, n)).mean(axis=1)
    print(f"n={n:>4}:  样本均值的 std (实测) = {means.std(ddof=1):6.3f}   "
          f"理论 SE = σ/√n = {sigma/np.sqrt(n):6.3f}")


<a id="3"></a>
## 3. √n 法则：精度的代价 / The √n Law

SE 随 $\sqrt{n}$ 缩小 ⇒ **精度翻倍要 4 倍数据，多一位小数要 100 倍数据**。

| 目标 | 需要的 n 倍数 |
|---|---|
| SE 减半 | ×4 |
| SE 降到 1/10 | ×100 |

这就是为什么 A/B 测试动辄要几十万样本（2.8 节算给你看），也是"大数据"对**均值类指标**边际价值递减的原因——从 1M 到 100M 样本，SE 只降 10 倍。
This is why A/B tests need huge samples, and why going from 1M to 100M rows shrinks SE only 10x — diminishing returns for mean-type metrics.


<a id="4"></a>
## 4. CLT 对偏态数据：n 要多大 / How Big Must n Be?

教科书"n ≥ 30"是**对称分布**的经验值。**偏态越重，需要的 n 越大**。

一个实用判据（Cochran / Sugden）：$n \gtrsim 28 + 25\,g_1^2$（$g_1$ = 偏度）。
- 对称（$g_1=0$）→ n≈30 够
- 中度右偏（$g_1=2$，指数分布）→ n≈130
- 重度右偏（$g_1=4$，某些收入数据）→ n≈430


In [ ]:
# 验证: 指数分布 (skew=2) 的样本均值什么时候"够正态"
# When does the mean of exponential data look normal enough?
fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
for ax, n in zip(axes, [5, 30, 130, 500]):
    means = rng.exponential(1.0, size=(20_000, n)).mean(axis=1)
    z = (means - 1.0) / (1.0/np.sqrt(n))            # 标准化 / standardize
    st.probplot(z, dist="norm", plot=ax)
    ax.get_lines()[0].set(markersize=2, alpha=0.4)
    ax.set_title(f"n={n}  (skew of mean: {st.skew(means):.2f})", fontsize=10)
plt.suptitle("CLT on exponential data: QQ plots of standardized means", y=1.04)
plt.tight_layout(); plt.show()


**n=30 时右尾仍明显翘起**（继承了母体的右偏）——"n≥30 万能"是迷思。**n=130 后才基本贴线**，与 $28+25 g_1^2 = 128$ 的判据吻合。

> 💡 实战含义：对重偏态指标（收入、GMV、时长）做 t 检验/置信区间，**小样本下名义 95% 实际可能只有 90%**。解法：加大 n、log 变换、或 bootstrap（2.11）。


<a id="5"></a>
## 5. 不止均值：中位数 / 分位数的 CLT / CLT Beyond the Mean

CLT 不是均值专属。**样本中位数**同样渐近正态：
$$\hat{m} \;\overset{\text{approx}}{\sim}\; \mathcal{N}\!\Big(m,\; \frac{1}{4n\,f(m)^2}\Big)$$
其中 $f(m)$ 是真实密度在中位数处的值。

**直觉**：中位数附近数据越密（$f(m)$ 大），中位数被"钉"得越牢。

正态母体下可推出 $\mathrm{SE}(\hat m) \approx 1.2533\,\sigma/\sqrt{n}$ —— 中位数比均值"贵" 25%（同精度需要 ~57% 更多样本），这是均值的**效率优势**；但 2.1 节讲过中位数的**稳健优势**。又是 trade-off。
For normal data the median needs ~57% more samples for the same precision — efficiency vs robustness, the eternal trade.


In [ ]:
# 验证中位数的渐近正态 + 1.2533 系数 / Verify the median CLT + the 1.2533 factor
n = 200
medians = np.median(rng.normal(0, 1, size=(40_000, n)), axis=1)
se_emp = medians.std(ddof=1)
se_theory = np.sqrt(np.pi/2) / np.sqrt(n)        # 1.2533/√n for σ=1

print(f"中位数 SE 实测   = {se_emp:.5f}")
print(f"理论 1.2533/√n  = {se_theory:.5f}")
print(f"均值的 SE        = {1/np.sqrt(n):.5f}   ← 中位数贵 {se_emp*np.sqrt(n)*100-100:.0f}%")
print(f"中位数分布偏度   = {st.skew(medians):.3f}  (≈0, 正态 ✓)")


<a id="6"></a>
## 6. ⚠ CLT 失效案例 1：最大值 / Failure Case 1: the Maximum

CLT 只管"**和/平均**"。**最大值**是另一套定律（极值理论，Fisher–Tippett）：归一化后收敛到 **Gumbel / Fréchet / Weibull** 三者之一，**永远不是正态**。
CLT governs sums/averages only. Maxima converge (after normalization) to Gumbel/Fréchet/Weibull — never normal.


In [ ]:
# 指数样本的最大值 → Gumbel，不是正态 / Max of exponentials → Gumbel
n = 1000
maxima = rng.exponential(1.0, size=(30_000, n)).max(axis=1)
z = maxima - np.log(n)                            # 标准化: max - ln(n) → Gumbel(0,1)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
xs = np.linspace(-3, 8, 300)
axes[0].hist(z, bins=100, density=True, alpha=0.7, label="max - ln(n)")
axes[0].plot(xs, st.gumbel_r.pdf(xs), "r-", lw=2, label="Gumbel(0,1)")
axes[0].plot(xs, st.norm.pdf(xs, z.mean(), z.std()), "g--", lw=2, label="best normal")
axes[0].legend(); axes[0].set_title("Maxima follow Gumbel — note the right skew")
st.probplot(z, dist="norm", plot=axes[1])
axes[1].get_lines()[0].set(markersize=2, alpha=0.4)
axes[1].set_title("Normal QQ: clear right-tail bend")
plt.tight_layout(); plt.show()
print(f"skew of maxima = {st.skew(z):.3f}  (Gumbel 理论值 ≈ 1.14, 正态应为 0)")


**右偏 1.1，永不消失**——n 再大也不会正态。

> 💡 实战含义：**P99 延迟、最大日跌幅、年度洪峰**这类"极值型指标"不能套均值的统计学——用极值理论（EVT）或分位数 bootstrap。"对 max 算 t 置信区间"是真实世界常见错误。
> P99 latency, max drawdown, flood peaks — extremes need EVT, not mean-based stats.


<a id="7"></a>
## 7. ⚠ CLT 失效案例 2：Cauchy（无均值）/ Failure Case 2: Cauchy

CLT 的前提是 $\sigma^2 < \infty$。**Cauchy 分布连均值都没有**（积分发散）——它的样本均值**永远是 Cauchy 本身**，n 再大也不收敛！
Cauchy has no mean at all. Its sample mean is Cauchy at every n — averaging buys you literally nothing.


In [ ]:
# 惊悚演示: Cauchy 的 running mean 永不收敛 / The horror: Cauchy running means never settle
fig, ax = plt.subplots(figsize=(9, 3.5))
for i in range(4):
    c = rng.standard_cauchy(100_000)
    ax.plot(np.cumsum(c) / np.arange(1, 100_001), lw=0.7, alpha=0.8, label=f"path {i+1}")
ax.set_xscale("log"); ax.axhline(0, color="k", lw=0.5)
ax.set_title("Cauchy running means: n=100,000 and still jumping — LLN/CLT both dead")
ax.legend(); plt.tight_layout(); plt.show()

# 数值证据: 均值的分布宽度不随 n 缩小
for n in [10, 1000, 100_000]:
    m = rng.standard_cauchy((2000, n)).mean(axis=1)
    print(f"n={n:>7}: 样本均值的 IQR = {np.percentile(m,75)-np.percentile(m,25):.3f}   (不缩小!)")


**IQR 在 n=10 和 n=100,000 时一样宽**——平均完全无效。每条路径还会被单个巨值"踹飞"。

> 💡 谁是现实里的 Cauchy？**两个正态变量的比值**（金融 beta 的某些估计）、物理共振峰。更广泛的教训：**重尾到 $\alpha$-stable 区间（无方差）的数据，均值是陷阱**——比特币早期日收益、城市人口增长率都接近这区。检查方法：画 running mean，不稳就警惕。
> Ratios of normals are Cauchy. Broader lesson: for α-stable heavy tails (infinite variance), the mean is a trap. Diagnostic: plot the running mean; if it won't settle, beware.


<a id="8"></a>
## 8. 实战：模拟器验证一切 / Hands-on: One Simulator to Verify It All

写一个**通用 CLT 实验器**：任何分布 × 任何统计量 → 抽样分布 + 正态性评分。把全课结论一表打尽。


In [ ]:
def sampling_distribution(sampler, statistic, n, n_sim=20_000, rng=rng):
    # sampler(size) -> 样本; statistic(2d array, axis=1) -> 每行一个统计量
    data = sampler((n_sim, n))
    return statistic(data, axis=1)

experiments = [
    # (名字, 抽样器, 统计量, n)
    ("Exp 均值, n=130",      lambda s: rng.exponential(1, s),  np.mean,   130),
    ("Exp 中位数, n=130",    lambda s: rng.exponential(1, s),  np.median, 130),
    ("Exp 最大值, n=130",    lambda s: rng.exponential(1, s),  np.max,    130),
    ("Cauchy 均值, n=130",   lambda s: rng.standard_cauchy(s), np.mean,   130),
    ("Cauchy 中位数, n=130", lambda s: rng.standard_cauchy(s), np.median, 130),
]

print(f"{'experiment':<22} {'skew':>7} {'ex_kurt':>9}   verdict")
print("-" * 60)
for name, sampler, stat, n in experiments:
    d = sampling_distribution(sampler, stat, n)
    sk, ku = st.skew(d), st.kurtosis(d)
    ok = "≈ normal ✓" if abs(sk) < 0.15 and abs(ku) < 0.3 else "NOT normal ✗"
    print(f"{name:<24} {sk:>7.2f} {ku:>9.2f}   {ok}")


**一张表浓缩全课**：
- Exp 均值 / 中位数 → 正态 ✓（CLT 及其分位数版都工作）
- Exp **最大值** → 右偏不消（Gumbel 域）
- Cauchy **均值** → 彻底疯（无方差，CLT 死）
- Cauchy **中位数** → 居然正态 ✓！——中位数的 CLT 只要密度 $f(m)>0$，**不要求方差存在**。**重尾数据用中位数**，这是第三次出现这个结论（2.1 稳健性、2.3 效率、现在是"存活能力"）。
- The Cauchy **median** is fine — quantile CLTs don't need finite variance. Third time this course lands on "use medians for heavy tails".


<a id="9"></a>
## 9. 小结 / Summary

```
LLN: 收敛方向 (μ)          速度: O(1/√n) (Chebyshev)
CLT: 收敛形状 (Normal)      条件: σ² < ∞, i.i.d.

标准误 SE = σ/√n ⭐
  ├── SD 描述数据, SE 描述估计量 — 别混
  ├── 精度翻倍 → 4 倍样本
  └── 偏态数据: n ≳ 28 + 25·skew²  ("n≥30" 只适用对称)

CLT 的边界:
  ├── 中位数/分位数: 有自己的 CLT, 且不需方差存在 ✓
  ├── 最大值: → Gumbel/Fréchet/Weibull (极值理论) ✗
  └── 无方差 (Cauchy/α-stable): 均值永不收敛 ✗ → 用中位数
```

### 💡 面试速查
1. **SD vs SE**：数据散布 vs 估计量散布；SE 随 √n 缩小
2. **样本翻倍精度提高 √2 倍**，不是 2 倍
3. **n≥30 的真相**：对称才够；skew=2 要 130+
4. **P99/max 不能用 t 区间**——极值不归 CLT 管
5. **running mean 不稳 = 重尾警报**，换中位数

### 下一节
**2.4 抽样方法**——SE 公式假设"简单随机抽样"；现实中怎么抽（分层/整群/水库）直接改变 SE，也埋着选择偏差的坑。
